In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.1 MB/s eta 0:00:0000:01


In [2]:
import os
import urllib.request
import gzip
import shutil
import networkx as nx

# Configuration
DATA_DIR = 'datasets'
os.makedirs(DATA_DIR, exist_ok=True)

print(f"==================================================")
print(f"   PREPARING SELECTED DATASETS: Polblogs, Facebook, Email")
print(f"==================================================\n")

# --- Helper: Download and Unzip .gz ---
def download_and_extract(url, final_filename):
    gz_path = os.path.join(DATA_DIR, f"{final_filename}.gz")
    final_path = os.path.join(DATA_DIR, final_filename)
    
    if os.path.exists(final_path):
        print(f"  -> Found {final_filename}, skipping.")
        return

    print(f"  -> Downloading {final_filename}...")
    try:
        urllib.request.urlretrieve(url, gz_path)
        with gzip.open(gz_path, 'rb') as f_in:
            with open(final_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        os.remove(gz_path)
        print(f"  -> [SUCCESS] Saved {final_filename}")
    except Exception as e:
        print(f"  -> [ERROR] Failed to download {final_filename}: {e}")

# 1. POLBLOGS (via PyG -> .txt to avoid MTX errors)
# -------------------------------------------------
print("[1/3] Processing Polblogs...")
pol_path = os.path.join(DATA_DIR, 'polblogs.txt')
if not os.path.exists(pol_path):
    try:
        from torch_geometric.datasets import PolBlogs
        from torch_geometric.utils import to_networkx
        print("  -> Fetching via torch_geometric...")
        dataset = PolBlogs(root='/tmp/PolBlogs')
        G = to_networkx(dataset[0], to_undirected=True)
        # Save as simple edge list
        nx.write_edgelist(G, pol_path, data=False)
        print(f"  -> [SUCCESS] Saved polblogs.txt")
    except Exception as e:
        print(f"  -> [ERROR] Could not fetch Polblogs: {e}")
else:
    print(f"  -> Found polblogs.txt, skipping.")

# 2. FACEBOOK (Direct SNAP Download)
# -------------------------------------------------
print("\n[2/3] Processing Facebook...")
download_and_extract(
    "https://snap.stanford.edu/data/facebook_combined.txt.gz",
    "facebook.txt"
)

# 3. EMAIL-EU-CORE (Direct SNAP Download)
# -------------------------------------------------
print("\n[3/3] Processing Email-Eu-Core...")
download_and_extract(
    "https://snap.stanford.edu/data/email-Eu-core.txt.gz",
    "email-Eu-core.txt"
)

print("\n--- DONE ---")

   PREPARING SELECTED DATASETS: Polblogs, Facebook, Email

[1/3] Processing Polblogs...
  -> Fetching via torch_geometric...


Extracting /tmp/PolBlogs/raw/polblogs.tar.gz
Processing...
Done!


  -> [SUCCESS] Saved polblogs.txt

[2/3] Processing Facebook...
  -> Downloading facebook.txt...
  -> [SUCCESS] Saved facebook.txt

[3/3] Processing Email-Eu-Core...
  -> Downloading email-Eu-core.txt...
  -> [SUCCESS] Saved email-Eu-core.txt

--- DONE ---


In [3]:
# -*- coding: utf-8 -*-

"""

INFLUENCE MINIMIZATION - ADVERSARY-FIRST VERSION (V6.1 - Continuous Reward)
WITH REAL-WORLD DATASETS

FEATURING:

- COMPETITIVE LINEAR THRESHOLD (LT) MODEL
- Continuous Zero-Sum Reward (R_B = 1 - 2*I_N, R_A = -R_B)
- REAL-WORLD DATASETS: Polblogs, Facebook, Email-Eu-Core
- Analysis and results saved to files for each experiment

This script runs all experiments and saves the raw .csv results to disk.
"""

import numpy as np
import networkx as nx
from collections import deque, Counter
import random
from scipy.optimize import linprog
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, Batch
import pandas as pd
import os
import warnings
from tqdm import tqdm
import math
from scipy import stats

warnings.filterwarnings('ignore')

# ============================================================================
# SETUP (REPRODUCIBILITY & GPU)
# ============================================================================
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print("=" * 100)
print("INFLUENCE MINIMIZATION: ADVERSARY-FIRST VERSION (V6.1 - Continuous Reward)")
print("WITH REAL-WORLD DATASETS: Polblogs, Facebook, Email-Eu-Core")
print(f"Device: {DEVICE} | CUDA: {torch.cuda.is_available()}")
print("MODEL: COMPETITIVE LINEAR THRESHOLD (LT)")
print("REWARD: Continuous Zero-Sum (R_B = 1 - 2*I_N)")
print("WIN CONDITION (for analysis): Blocker if I_N < 0.5 | Adversary if I_N > 0.5")
print("=" * 100)

# ============================================================================
# 7 OPTIMIZED META-STRATEGIES & CONSTANTS
# ============================================================================
META_STRATEGY_NAMES = [
    'HUB_TARGETING', 'BRIDGE_TARGETING', 'CLOSENESS_CENTRALITY', 'K_CORE_TARGETING',
    'PERIPHERAL_TARGETING', 'HIGH_CLUSTERING', 'OPPONENT_NEIGHBOR'
]
NUM_STRATEGIES = len(META_STRATEGY_NAMES)
MAX_ROUNDS = 30

# ============================================================================
# OPTIMIZED NEURAL NETWORKS
# ============================================================================
class OptimizedGATEncoder(nn.Module):
    """
    Graph Attention Network (GAT) encoder for node features.
    Uses 2 GAT layers with ELU activation and dropout.
    """
    def __init__(self, node_features=4, hidden_dim=64, heads=4, dropout_rate=0.1):
        super(OptimizedGATEncoder, self).__init__()
        self.dropout_rate = dropout_rate
        # Input features -> hidden_dim * heads
        self.conv1 = GATConv(node_features, hidden_dim, heads=heads, concat=True, dropout=dropout_rate)
        # hidden_dim * heads -> hidden_dim (final embedding dimension)
        self.conv2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout_rate)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Apply first GAT layer with ELU activation
        x = F.elu(self.conv1(x, edge_index))
        # Dropout applied internally by GATConv if dropout > 0
        # Apply second GAT layer with ELU activation
        x = F.elu(self.conv2(x, edge_index))
        return x

class OptimizedMetaStrategyNetwork(nn.Module):
    """
    Main DQN combining GAT encoder and MLP for Q-value output.
    Includes round information in the MLP input.
    """
    def __init__(self, node_features=4, gat_hidden=64, heads=4, mlp_hidden1=256, mlp_hidden2=128, dropout_rate=0.1):
        super(OptimizedMetaStrategyNetwork, self).__init__()
        self.gat = OptimizedGATEncoder(node_features, gat_hidden, heads, dropout_rate)
        # MLP input size = GAT output dimension + 1 (for round info)
        mlp_input_size = gat_hidden + 1
        self.strategy_network = nn.Sequential(
            nn.Linear(mlp_input_size, mlp_hidden1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(mlp_hidden1, mlp_hidden2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(mlp_hidden2, NUM_STRATEGIES) # Output Q-values for each strategy
        )

    def forward(self, data):
        # 1. Get node embeddings from GAT
        node_embeddings = self.gat(data)

        # 2. Pool node embeddings to get graph embedding
        if hasattr(data, 'batch') and data.batch is not None:
            # Batched graph: use global mean pool with batch indices
            graph_embedding = global_mean_pool(node_embeddings, data.batch)
            # Extract round info (feature index 3) for the first node of each graph in the batch
            start_indices = data.ptr[:-1]
            round_info_per_graph = data.x[start_indices, 3]
            round_info = round_info_per_graph.unsqueeze(1) # Shape: [batch_size, 1]
        else:
            # Single graph: create a dummy batch index tensor of zeros
            batch_index = torch.zeros(data.num_nodes, dtype=torch.long, device=node_embeddings.device)
            graph_embedding = global_mean_pool(node_embeddings, batch_index)
            # Extract round info from the first node (assuming it's representative)
            round_scalar = data.x[0, 3] if data.x.size(0) > 0 else torch.tensor(0.0, device=graph_embedding.device)
            round_info = round_scalar.unsqueeze(0).unsqueeze(0) # Shape: [1, 1]

        # 3. Combine graph embedding and round info
        # Ensure round_info is correctly shaped [batch_size, 1] before concatenation
        if round_info.dim() == 1: round_info = round_info.unsqueeze(1)
        elif round_info.dim() == 0: round_info = round_info.view(1, 1)

        # Handle potential shape mismatches (e.g., if batch has size 1 but treated differently)
        if graph_embedding.size(0) != round_info.size(0):
            if round_info.size(0) == 1 and graph_embedding.size(0) > 1:
                # Broadcast single round info across the batch
                round_info = round_info.expand(graph_embedding.size(0), -1)
            else:
                # If shapes still mismatch, log warning or raise error
                print(f"Warning: Shape mismatch during forward pass. GraphEmb: {graph_embedding.shape}, RoundInfo: {round_info.shape}. Using mean round info as fallback.")
                # Fallback: Use mean round info if shapes are incompatible (should ideally not happen)
                mean_round = data.x[:, 3].mean().item() if data.x.size(0) > 0 else 0.0
                round_info = torch.full((graph_embedding.size(0), 1), mean_round, device=graph_embedding.device)

        combined_features = torch.cat([graph_embedding, round_info], dim=1)

        # 4. Pass through MLP to get Q-values
        strategy_q_values = self.strategy_network(combined_features)

        # Return Q-values and the detached graph embedding (useful for analysis)
        return strategy_q_values, graph_embedding.detach()


# ============================================================================
# OPTIMIZED ENVIRONMENT (LT MODEL) WITH CONTINUOUS REWARD
# ============================================================================
class OptimizedCompetitiveLTEnvironment:
    """
    The game environment. Manages graph state, actions, and simulates LT influence.
    Uses a continuous zero-sum reward function.
    """
    def __init__(self, graph_data):
        """Initializes the environment with a specific graph and precomputes metrics."""
        self.base_graph = graph_data['graph']
        self.name = graph_data['name']
        self.num_nodes = self.base_graph.number_of_nodes()

        self._precompute_essential_metrics()

        self.current_round = 0
        self.influence_history = []
        self.last_influenced_size = 0.0

    def _precompute_essential_metrics(self):
        """
        Calculates and stores node-level graph metrics at the start.
        """
        if self.num_nodes == 0:
            self.degree, self.betweenness, self.closeness, self.k_core, self.clustering = [{},{},{},{},{}]
            return

        self.degree = dict(self.base_graph.degree())
        # Ensure minimum degree of 1 for LT calculation stability
        for node, deg in self.degree.items():
            if deg == 0:
                self.degree[node] = 1

        # Use sampling for large graphs if needed, but for ~1k nodes, full calculation is feasible
        k_sample = min(self.num_nodes, 500) # Limit sampling if desired
        self.betweenness = nx.betweenness_centrality(self.base_graph, normalized=True, seed=42)
        self.closeness = nx.closeness_centrality(self.base_graph)

        try:
            self.k_core = nx.core_number(self.base_graph)
        except Exception: # Handle potential errors with core_number calculation
            print(f"Warning: Could not compute k-core for {self.name}. Defaulting to 0.")
            self.k_core = {n: 0 for n in self.base_graph.nodes()}

        self.clustering = nx.clustering(self.base_graph)

    def reset(self):
        """Resets the environment for a new episode."""
        nodes = list(self.base_graph.nodes())
        self.node_mapping = {env_idx: graph_node for env_idx, graph_node in enumerate(nodes)}
        self.reverse_mapping = {graph_node: env_idx for env_idx, graph_node in self.node_mapping.items()}

        self.occupied = {0: set(), 1: set()} # 0: Adversary, 1: Blocker
        self.free_nodes = set(self.base_graph.nodes())

        self.current_round = 0
        self.influence_history = []
        self.last_influenced_size = 0.0

        # Pre-create the edge index tensor if not already done
        if not hasattr(self, '_edge_index'):
            edge_list = []
            for u, v in self.base_graph.edges():
                env_u, env_v = self.reverse_mapping.get(u), self.reverse_mapping.get(v)
                if env_u is not None and env_v is not None:
                    edge_list.append((env_u, env_v))
                    edge_list.append((env_v, env_u)) # Undirected graph

            if not edge_list:
                self._edge_index = torch.empty((2, 0), dtype=torch.long)
            else:
                # Ensure edge indices are within bounds [0, num_nodes-1]
                max_idx = self.num_nodes - 1
                valid_edges = [(u, v) for u, v in edge_list if 0 <= u <= max_idx and 0 <= v <= max_idx]
                if valid_edges:
                    self._edge_index = torch.tensor(valid_edges, dtype=torch.long).t().contiguous()
                else:
                    self._edge_index = torch.empty((2, 0), dtype=torch.long)

        return self.get_state()

    def get_available_nodes(self):
        """Returns a list of node indices that are not yet occupied."""
        # Use list comprehension for potentially faster creation
        return [self.reverse_mapping[n] for n in self.free_nodes if n in self.reverse_mapping]

    def select_seed(self, player, node):
        """
        Assigns a selected node to a player (0 or 1) and updates state.
        """
        if node is None:
            return False, None

        graph_node = self.node_mapping.get(node)
        # Check if node exists and is free using set lookups (O(1) average)
        if graph_node in self.free_nodes:
            self.occupied[player].add(graph_node)
            self.free_nodes.remove(graph_node) # Use remove as we know it exists
            return True, graph_node
        return False, None

    def fast_propagate_lt(self, num_simulations=5):
        """
        Performs influence propagation using the Linear Threshold (LT) model.
        Optimized implementation.
        """
        total_influenced_size = 0
        seed_set = self.occupied[0]
        immunized_set = self.occupied[1]

        if not seed_set or self.num_nodes == 0:
            self.last_influenced_size = 0.0
            self.influence_history.append(0.0)
            return 0.0

        # Pre-filter seeds: only consider non-immunized seeds
        initial_seeds = seed_set - immunized_set
        if not initial_seeds:
            self.last_influenced_size = 0.0
            self.influence_history.append(0.0)
            return 0.0

        # Pre-calculate inverse degrees for faster weight lookup
        # Ensure degrees are not zero during inversion
        inv_degrees = {node: 1.0 / max(1, self.degree.get(node, 1)) for node in self.base_graph.nodes()}

        for _ in range(num_simulations):
            thresholds = {n: random.random() for n in self.base_graph.nodes()}
            activated = initial_seeds.copy()
            newly_activated = initial_seeds.copy()

            while newly_activated:
                current_round_activated = set()
                # Consider only neighbors of *newly* activated nodes as candidates
                nodes_to_check = set()
                for u in newly_activated:
                    # Optimized neighbor fetching
                    try:
                        neighbors = list(self.base_graph.adj[u]) if u in self.base_graph else []
                        for v in neighbors:
                            if v not in activated and v not in immunized_set:
                                nodes_to_check.add(v)
                    except KeyError: # Handle nodes potentially not in graph
                        continue

                for v in nodes_to_check:
                    # Calculate incoming weight efficiently
                    total_weight = 0.0
                    try:
                        # Optimized neighbor check and weight sum
                        node_v_inv_degree = inv_degrees.get(v, 1.0) # Default to 1 if not found
                        neighbors_v = list(self.base_graph.adj[v]) if v in self.base_graph else []
                        for u_neighbor in neighbors_v:
                            if u_neighbor in activated:
                                total_weight += node_v_inv_degree # Use precomputed inverse degree
                    except KeyError:
                        continue

                    # Check against threshold
                    if total_weight >= thresholds.get(v, 1.1): # Thresholds > 1 are impossible to reach
                        current_round_activated.add(v)

                newly_activated = current_round_activated
                activated.update(newly_activated)

            total_influenced_size += len(activated)

        expected_influenced_size = total_influenced_size / num_simulations
        self.last_influenced_size = expected_influenced_size
        self.influence_history.append(expected_influenced_size) # Storing absolute for now
        return expected_influenced_size

    def get_state(self):
        """
        Generates the current graph state as a PyTorch Geometric Data object.
        Features: [is_free, is_adversary, is_blocker, norm_round]
        """
        if self.num_nodes == 0:
            # Return empty tensors with correct shapes and types
            return Data(x=torch.zeros((0, 4), dtype=torch.float32),
                        edge_index=torch.zeros((2, 0), dtype=torch.long))

        # Initialize features efficiently
        x = torch.zeros((self.num_nodes, 4), dtype=torch.float32)
        x[:, 0] = 1.0 # Assume all free initially

        # Map occupied nodes - use reverse mapping indices directly
        # Ensure nodes exist in mapping before getting index
        adv_indices = [self.reverse_mapping[n] for n in self.occupied[0] if n in self.reverse_mapping]
        blocker_indices = [self.reverse_mapping[n] for n in self.occupied[1] if n in self.reverse_mapping]

        # Use advanced indexing for efficiency if indices lists are not empty
        if adv_indices:
            adv_indices_tensor = torch.tensor(adv_indices, dtype=torch.long)
            x[adv_indices_tensor, 0] = 0.0 # Not free
            x[adv_indices_tensor, 1] = 1.0 # Is adversary

        if blocker_indices:
            blocker_indices_tensor = torch.tensor(blocker_indices, dtype=torch.long)
            x[blocker_indices_tensor, 0] = 0.0 # Not free
            x[blocker_indices_tensor, 2] = 1.0 # Is blocker

        # Add normalized round info (broadcast)
        # Ensure MAX_ROUNDS > 0 to avoid division by zero
        norm_round = self.current_round / max(1, MAX_ROUNDS)
        x[:, 3] = norm_round

        # Ensure edge_index is valid before creating Data object
        if not hasattr(self, '_edge_index') or not isinstance(self._edge_index, torch.Tensor):
            # Handle case where reset/edge_index creation failed
            print(f"Warning: Edge index invalid or missing for graph {self.name}. Using empty edge index.")
            edge_index = torch.empty((2, 0), dtype=torch.long)
        else:
            edge_index = self._edge_index

        return Data(x=x, edge_index=edge_index)

    def calculate_rewards(self, previous_influence, done=False):
        """
        Calculates rewards based on the change in influence (intermediate)
        or the final continuous zero-sum outcome (final).

        REVISED: Uses continuous zero-sum final reward R_B = 1 - 2*I_N.
        """
        current_influence = self.last_influenced_size
        # Ensure num_nodes > 0 before division
        normalized_influence = current_influence / self.num_nodes if self.num_nodes > 0 else 0.0
        normalized_influence = np.clip(normalized_influence, 0.0, 1.0) # Ensure it's within [0, 1]

        # Intermediate reward: Based on the change from the previous step
        influence_change = current_influence - previous_influence
        normalized_change = influence_change / self.num_nodes if self.num_nodes > 0 else 0.0

        adversary_intermediate_reward = normalized_change
        blocker_intermediate_reward = -normalized_change

        if done:
            # Final Reward: Continuous Zero-Sum
            # Blocker Reward: R_B = 1 - (2 * I_N) -> Ranges from 1 (0% spread) to -1 (100% spread)
            # Adversary Reward: R_A = -R_B = (2 * I_N) - 1 -> Ranges from -1 (0% spread) to 1 (100% spread)
            blocker_final_reward = 1.0 - (2.0 * normalized_influence)
            adversary_final_reward = -blocker_final_reward

            # Clamp rewards to [-1, 1] just in case normalization had issues
            blocker_final_reward = np.clip(blocker_final_reward, -1.0, 1.0)
            adversary_final_reward = np.clip(adversary_final_reward, -1.0, 1.0)

            return adversary_final_reward, blocker_final_reward
        else:
            # Return intermediate rewards for non-terminal steps
            return adversary_intermediate_reward, blocker_intermediate_reward

    def execute_meta_strategy(self, strategy_id, player):
        """
        Selects the best available node based on the chosen meta-strategy heuristic.
        Includes tie-breaking and fallbacks.
        """
        free_nodes_env_indices = self.get_available_nodes()
        if not free_nodes_env_indices:
            return None

        # Convert env indices back to original graph node IDs efficiently using a set for O(1) lookups
        free_nodes_graph = {self.node_mapping[i] for i in free_nodes_env_indices}

        best_node_graph = None

        # Define metric getters concisely using lambda functions or direct dict access
        metrics_map = {
            0: self.degree,       # HUB_TARGETING (Maximize)
            1: self.betweenness,  # BRIDGE_TARGETING (Maximize)
            2: self.closeness,     # CLOSENESS_CENTRALITY (Maximize)
            3: self.k_core,       # K_CORE_TARGETING (Maximize)
            4: self.degree,       # PERIPHERAL_TARGETING (Minimize degree)
            5: self.clustering,   # HIGH_CLUSTERING (Maximize)
            6: None               # OPPONENT_NEIGHBOR (Special case handled below)
        }

        target_metric_dict = metrics_map.get(strategy_id)

        if strategy_id == 6: # OPPONENT_NEIGHBOR
            opponent_nodes = self.occupied[1 - player]
            if not opponent_nodes: # Fallback to hub targeting if opponent hasn't moved
                strategy_id = 0 # Act as if it's hub targeting now
                target_metric_dict = self.degree
            else:
                # Calculate opponent neighbor scores for free nodes
                scores = {}
                for node in free_nodes_graph:
                    try:
                        # Count neighbors efficiently using set intersection or list comprehension
                        neighbors = set(self.base_graph.adj.get(node, [])) # Get neighbors, default empty if node missing
                        opp_neighbor_count = len(neighbors.intersection(opponent_nodes))
                        scores[node] = opp_neighbor_count
                    except Exception: # Catch potential errors with neighbor access
                        scores[node] = 0 # Assign score 0 if error occurs

                if scores:
                    max_score = max(scores.values())
                    # Get all nodes with the max score for tie-breaking
                    candidates = [node for node, score in scores.items() if score == max_score]
                    if len(candidates) == 1:
                        best_node_graph = candidates[0]
                    else:
                        # Tie-break: highest degree among candidates
                        best_node_graph = max(candidates, key=lambda n: self.degree.get(n, 0))

        # Handle strategies 0-5
        if target_metric_dict is not None and strategy_id != 6: # Check strategy_id again in case it changed in opp_neighbor fallback
            is_minimize = (strategy_id == 4) # Peripheral targeting minimizes degree
            best_score = float('inf') if is_minimize else -float('inf')
            candidates = [] # Store candidates for tie-breaking

            for node in free_nodes_graph:
                score = target_metric_dict.get(node, 0) # Default score is 0

                if is_minimize:
                    if score < best_score:
                        best_score = score
                        candidates = [node] # New best, reset candidates
                    elif score == best_score:
                        candidates.append(node) # Add to tie candidates
                else: # Maximization
                    if score > best_score:
                        best_score = score
                        candidates = [node]
                    elif score == best_score:
                        candidates.append(node)

            # Select from candidates
            if len(candidates) == 1:
                best_node_graph = candidates[0]
            elif len(candidates) > 1:
                # Apply tie-breaking rules based on strategy
                if strategy_id == 0: # HUB_TARGETING tie-break: highest betweenness
                    best_node_graph = max(candidates, key=lambda n: self.betweenness.get(n, 0))
                elif strategy_id == 4: # PERIPHERAL_TARGETING tie-break: lowest betweenness
                    best_node_graph = min(candidates, key=lambda n: self.betweenness.get(n, 1.0)) # Default 1 if missing
                else: # Default tie-break: highest degree (or choose randomly if degrees also tie)
                    best_node_graph = max(candidates, key=lambda n: self.degree.get(n, 0))

        # Fallback: If no best node found after all logic, choose randomly from available
        if best_node_graph is None and free_nodes_graph:
            best_node_graph = random.choice(list(free_nodes_graph))

        # Return the environment index of the chosen node
        return self.reverse_mapping.get(best_node_graph) if best_node_graph is not None else None


# ============================================================================
# NASH EQUILIBRIUM SOLVER
# ============================================================================
def solve_zero_sum_nash(payoff_matrix):
    """
    Solves a two-player, zero-sum game given its payoff matrix using linear programming.
    Returns optimal mixed strategies for row (player 1) and column (player 2) players,
    and the value of the game for the row player. Uses 'highs' solver.
    """
    if not isinstance(payoff_matrix, np.ndarray) or payoff_matrix.size == 0:
        num_actions = NUM_STRATEGIES # Assume standard number if empty
        uniform = np.ones(num_actions) / num_actions if num_actions > 0 else np.array([])
        return uniform, uniform, 0.0

    n_row_actions, n_col_actions = payoff_matrix.shape
    if n_row_actions == 0 or n_col_actions == 0:
        return np.array([]), np.array([]), 0.0

    # Ensure matrix contains finite numbers
    if not np.all(np.isfinite(payoff_matrix)):
        row_strategy = np.ones(n_row_actions) / n_row_actions if n_row_actions > 0 else np.array([])
        col_strategy = np.ones(n_col_actions) / n_col_actions if n_col_actions > 0 else np.array([])
        return row_strategy, col_strategy, 0.0 # Cannot determine value reliably

    # Shift payoff matrix to ensure all entries are positive for LP standard form
    min_val = np.min(payoff_matrix)
    shift = max(0, -min_val) + 1e-6 # Add epsilon for strict positivity
    shifted_payoff = payoff_matrix + shift

    # --- Row Player (Maximizer) ---
    # We want to maximize v subject to: v * ones - payoff_matrix * col_strategy <= 0
    # Equivalent to minimizing 1/v = sum(x) subject to: -shifted_payoff^T * x <= -ones, x >= 0
    c_row = np.ones(n_row_actions)
    A_ub_row = -shifted_payoff.T
    b_ub_row = -np.ones(n_col_actions)
    bounds_row = [(0, None)] * n_row_actions # x >= 0

    try:
        result_row = linprog(c_row, A_ub=A_ub_row, b_ub=b_ub_row, bounds=bounds_row, method='highs')

        if result_row.success and result_row.fun > 1e-9: # Check if 1/v is positive
            shifted_game_value = 1.0 / result_row.fun
            game_value = shifted_game_value - shift # Original game value
            row_strategy = result_row.x * shifted_game_value

            # --- Column Player (Minimizer) ---
            c_col = -np.ones(n_col_actions) # Maximize sum(y) = 1/v'
            A_ub_col = shifted_payoff # shifted_payoff * y <= ones
            b_ub_col = np.ones(n_row_actions)
            bounds_col = [(0, None)] * n_col_actions

            result_col = linprog(c_col, A_ub=A_ub_col, b_ub=b_ub_col, bounds=bounds_col, method='highs')

            if result_col.success and result_col.fun < -1e-9: # Max sum(y) = -min(-sum(y)) > 0
                shifted_game_value_col = -1.0 / result_col.fun
                col_strategy = result_col.x * shifted_game_value_col

                # --- Normalization and Cleanup ---
                row_strategy = np.maximum(0, row_strategy) # Ensure non-negative
                col_strategy = np.maximum(0, col_strategy)
                row_sum = np.sum(row_strategy)
                col_sum = np.sum(col_strategy)

                if row_sum > 1e-6:
                    row_strategy /= row_sum
                else: # Fallback if sum is near zero
                    row_strategy = np.ones(n_row_actions) / n_row_actions if n_row_actions > 0 else np.array([])

                if col_sum > 1e-6:
                    col_strategy /= col_sum
                else: # Fallback
                    col_strategy = np.ones(n_col_actions) / n_col_actions if n_col_actions > 0 else np.array([])

                return row_strategy, col_strategy, game_value # Return original game value from row player solution
            else:
                # Column solver failed or gave non-positive value
                raise Exception("Column LP solver failed or yielded non-positive value.")

        else:
            # Row solver failed or gave non-positive value
            raise Exception("Row LP solver failed or yielded non-positive value.")

    except Exception as e:
        # Fallback to uniform strategy
        row_strategy = np.ones(n_row_actions) / n_row_actions if n_row_actions > 0 else np.array([])
        col_strategy = np.ones(n_col_actions) / n_col_actions if n_col_actions > 0 else np.array([])
        # Estimate game value as mean payoff under uniform strategies
        game_value = np.mean(payoff_matrix) if payoff_matrix.size > 0 else 0.0
        return row_strategy, col_strategy, game_value


# ============================================================================
# OPTIMIZED META-STRATEGY DQN AGENTS
# ============================================================================
class OptimizedMetaStrategyDQNAgent:
    """
    Agent implementation using DQN or Minimax-DQN for meta-strategy selection.
    """
    def __init__(self, is_blocker, use_minimax=True, lr=1e-4, gamma=0.95, target_update_freq=100, batch_size=64, buffer_size=20000):
        self.is_blocker = is_blocker
        self.use_minimax = use_minimax
        self.gamma = gamma
        self.target_update_freq = target_update_freq
        self.update_count = 0

        # Network setup
        self.network = OptimizedMetaStrategyNetwork().to(DEVICE)
        self.target_network = OptimizedMetaStrategyNetwork().to(DEVICE)
        self.target_network.load_state_dict(self.network.state_dict())
        self.target_network.eval() # Target network is only for inference

        # Optimizer
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr, weight_decay=1e-5) # Added weight decay

        # Replay Buffer
        self.replay_buffer = deque(maxlen=buffer_size)
        self.batch_size = batch_size

        self.name = f"{'Minimax' if use_minimax else 'Indep'}-DQN ({'Blocker' if is_blocker else 'Adversary'})"

    def get_q_values(self, state, use_target=False):
        """Gets Q-values for a given state using the specified network."""
        net = self.target_network if use_target else self.network
        is_training = net.training # Store current mode
        net.eval() # Set to evaluation mode for inference

        # Ensure state is a PyG Data object and on the correct device
        if not isinstance(state, Data):
            pass # Assuming state is already a Data object
        state = state.to(DEVICE)

        with torch.no_grad():
            try:
                q_values_tensor, graph_emb = net(state) # Assumes network returns (q_values, graph_embedding)
                # Ensure q_values_tensor is detached and moved to CPU
                q_values_np = q_values_tensor.squeeze().detach().cpu().numpy()
            except Exception as e:
                print(f"Error during network forward pass: {e}")
                print(f"State details: Nodes={state.num_nodes}, Edges={state.num_edges}, Batch={state.batch}")
                # Fallback to zeros or handle error appropriately
                q_values_np = np.zeros(NUM_STRATEGIES)
                graph_emb = torch.zeros(self.network.gat.conv2.out_channels, device=DEVICE) # Match embedding size

        # Handle numpy conversion edge cases (scalar output)
        if q_values_np.ndim == 0:
            q_values_np = np.array([q_values_np.item()]) # Convert scalar to 1D array

        # Ensure correct shape (pad if necessary, though ideally shouldn't happen)
        current_len = len(q_values_np)
        if current_len < NUM_STRATEGIES:
            q_values_np = np.pad(q_values_np, (0, NUM_STRATEGIES - current_len), 'constant', constant_values=-np.inf) # Pad with -inf

        if is_training: # Restore original mode if needed
            net.train()

        # Ensure graph_emb is detached
        graph_emb = graph_emb.detach()

        return q_values_np[:NUM_STRATEGIES], graph_emb # Return only the required number of Q-values

    def select_action(self, state, opponent_agent=None, epsilon=0.1):
        """
        Selects a meta-strategy using epsilon-greedy (for exploration),
        Minimax-DQN (solving Nash equilibrium), or Independent-DQN (greedy).
        Returns: strategy_id, graph_embedding, q_values, mixed_strategy (if minimax), game_value (if minimax)
        """
        # Epsilon-greedy exploration
        if random.random() < epsilon:
            strategy_id = random.randint(0, NUM_STRATEGIES - 1)
            # Get Q-values and embedding even for random choice for potential logging/analysis
            my_q_values, graph_emb = self.get_q_values(state)
            # Estimate value as average Q for random choice
            estimated_value = np.mean(my_q_values) if my_q_values.size > 0 and np.all(np.isfinite(my_q_values)) else 0.0
            return strategy_id, graph_emb, my_q_values, None, estimated_value

        # Exploitation phase
        my_q_values, graph_emb = self.get_q_values(state)

        # Handle potential non-finite Q-values before proceeding
        if not np.all(np.isfinite(my_q_values)):
            valid_indices = np.where(np.isfinite(my_q_values))[0]
            if len(valid_indices) > 0:
                strategy_id = np.random.choice(valid_indices)
                action_value = my_q_values[strategy_id]
            else: # All Q-values are invalid, choose completely random
                strategy_id = random.randint(0, NUM_STRATEGIES - 1)
                action_value = 0.0 # Cannot determine value
            return strategy_id, graph_emb, my_q_values, None, action_value

        if self.use_minimax and opponent_agent is not None:
            # Minimax-DQN: Solve the zero-sum game
            opp_q_values, _ = opponent_agent.get_q_values(state) # Opponent's Q-values for the same state

            # Ensure opponent Q-values are also finite
            if not np.all(np.isfinite(opp_q_values)):
                strategy_id = np.argmax(my_q_values)
                action_value = my_q_values[strategy_id]
                return strategy_id, graph_emb, my_q_values, None, action_value

            # Build the payoff matrix (row player's perspective)
            payoff_matrix = np.zeros((NUM_STRATEGIES, NUM_STRATEGIES))
            for i in range(NUM_STRATEGIES):
                for j in range(NUM_STRATEGIES):
                    # Payoff = My_Q - Opponent_Q (as used in previous logic)
                    payoff_matrix[i, j] = my_q_values[i] - opp_q_values[j]

            # Solve for Nash Equilibrium
            row_strategy, _, game_value = solve_zero_sum_nash(payoff_matrix)

            # Handle solver failures (e.g., NaN strategy)
            if np.isnan(row_strategy).any() or np.sum(row_strategy) < 1e-6: # Check sum too
                strategy_id = np.argmax(my_q_values)
                action_value = my_q_values[strategy_id]
                return strategy_id, graph_emb, my_q_values, None, action_value

            # Sample action from the row player's mixed strategy
            try:
                # Ensure probabilities sum exactly to 1 (handle potential floating point issues)
                row_strategy /= np.sum(row_strategy)
                strategy_id = np.random.choice(NUM_STRATEGIES, p=row_strategy)
            except ValueError: # If probabilities don't sum to 1 or are invalid
                strategy_id = np.argmax(my_q_values)
                action_value = my_q_values[strategy_id]
                return strategy_id, graph_emb, my_q_values, None, action_value

            # Return chosen strategy, embedding, Q-values, the mixed strategy itself, and the game value
            return strategy_id, graph_emb, my_q_values, row_strategy, game_value

        else:
            # Independent-DQN: Choose the action with the highest Q-value (greedy)
            strategy_id = np.argmax(my_q_values)
            # Value is simply the Q-value of the chosen action
            action_value = my_q_values[strategy_id]
            return strategy_id, graph_emb, my_q_values, None, action_value

    def store_transition(self, state, strategy_id, reward, next_state, done):
        """Stores experience tuple in the replay buffer. Clones tensors."""
        try:
            # Ensure tensors are cloned and detached if necessary
            # Storing directly on CPU might save GPU memory if buffer is large
            cpu_state = Data(x=state.x.clone().cpu(), edge_index=state.edge_index.clone().cpu())
            cpu_next_state = Data(x=next_state.x.clone().cpu(), edge_index=next_state.edge_index.clone().cpu())
            self.replay_buffer.append((cpu_state, strategy_id, reward, cpu_next_state, done))
        except Exception as e:
            print(f"Error storing transition: {e}")
            print(f"State: Nodes={state.num_nodes}, Edges={state.num_edges}")
            print(f"Next State: Nodes={next_state.num_nodes}, Edges={next_state.num_edges}")

    def update(self, opponent_agent=None):
        """Samples a batch, calculates targets, and performs a gradient update."""
        if len(self.replay_buffer) < self.batch_size:
            return 0.0, 0.0 # Not enough samples

        # Sample a batch
        batch = random.sample(self.replay_buffer, self.batch_size)
        states, strategy_ids, rewards, next_states, dones_list = zip(*batch)

        # Convert to tensors
        # Handle potential errors during conversion (e.g., if reward is not a number)
        try:
            rewards_tensor = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)
            dones_tensor = torch.tensor(dones_list, dtype=torch.float32, device=DEVICE) # Use float for (1-dones)
            strategy_ids_tensor = torch.tensor(strategy_ids, dtype=torch.long, device=DEVICE)
        except Exception as e:
            print(f"Error converting batch elements to tensors: {e}")
            return 0.0, 0.0

        # Batch states and next_states using PyG Batch
        try:
            # Move data to device *within* the list comprehension for Batching
            batched_states = Batch.from_data_list([s.to(DEVICE) for s in states])
            batched_next_states = Batch.from_data_list([ns.to(DEVICE) for ns in next_states])
        except Exception as e:
            print(f"Error creating PyG batch: {e}")
            return 0.0, 0.0

        # --- Calculate current Q-values ---
        self.network.train() # Ensure network is in training mode
        try:
            current_q_all, _ = self.network(batched_states)
            # Select the Q-value for the action taken
            current_q = current_q_all.gather(1, strategy_ids_tensor.unsqueeze(1)).squeeze(1)
        except Exception as e:
            print(f"Error during current Q calculation: {e}")
            return 0.0, 0.0

        # --- Calculate target Q-values ---
        target_q_val_list = [] # To store target values for each item in the batch
        avg_next_state_value = 0.0 # For logging/monitoring

        # Use target network(s) in eval mode and ensure no gradients are computed for targets
        with torch.no_grad():
            self.target_network.eval()
            if opponent_agent:
                opponent_agent.target_network.eval()

            if self.use_minimax and opponent_agent is not None:
                # Minimax-DQN: Target is reward + gamma * NashValue(next_state)
                try:
                    # Get Q-values for the entire batch of next states from target networks
                    next_q_all_mine_target, _ = self.target_network(batched_next_states)
                    next_q_all_opp_target, _ = opponent_agent.target_network(batched_next_states)
                except Exception as e:
                    print(f"Error during target Q calculation (Minimax): {e}")
                    return 0.0, 0.0

                next_state_values = []
                for i in range(self.batch_size):
                    if dones_list[i]:
                        target_q_val_list.append(rewards_tensor[i].item()) # Target is just the reward if done
                        next_state_values.append(0.0) # No next state value
                    else:
                        # Extract Q-values for the i-th next_state from the batched output
                        my_q_next = next_q_all_mine_target[i].cpu().numpy()
                        opp_q_next = next_q_all_opp_target[i].cpu().numpy()

                        # Check for non-finite Q-values before solving Nash
                        if not np.all(np.isfinite(my_q_next)) or not np.all(np.isfinite(opp_q_next)):
                            nash_value = 0.0 # Assign a neutral value
                        else:
                            # Build payoff matrix for the next state
                            payoff_matrix = np.zeros((NUM_STRATEGIES, NUM_STRATEGIES))
                            for r in range(NUM_STRATEGIES):
                                for c in range(NUM_STRATEGIES):
                                    if self.is_blocker: payoff_matrix[r, c] = my_q_next[r] - opp_q_next[c]
                                    else: payoff_matrix[r, c] = my_q_next[r] - opp_q_next[c]

                            # Solve Nash Equilibrium for the next state
                            _, _, nash_value = solve_zero_sum_nash(payoff_matrix)
                            # Handle potential NaN from solver
                            if np.isnan(nash_value):
                                nash_value = 0.0

                        target_q = rewards_tensor[i] + self.gamma * nash_value
                        target_q_val_list.append(target_q.item())
                        next_state_values.append(nash_value)

                # Calculate average next state value for logging (excluding terminal states)
                valid_next_values = [v for i, v in enumerate(next_state_values) if not dones_list[i]]
                avg_next_state_value = np.mean(valid_next_values) if valid_next_values else 0.0

            else:
                # Independent-DQN: Target is reward + gamma * max(Q_target(next_state))
                try:
                    next_q_all_target, _ = self.target_network(batched_next_states)
                    next_q_max = next_q_all_target.max(1)[0] # Get max Q-value for each next state
                except Exception as e:
                    print(f"Error during target Q calculation (Independent): {e}")
                    return 0.0, 0.0

                # Calculate target Q: R + gamma * max_a' Q_target(s', a') * (1 - done)
                target_q_tensor = rewards_tensor + self.gamma * next_q_max * (1.0 - dones_tensor)
                target_q_val_list = target_q_tensor.tolist() # Convert tensor to list

                # Calculate average next state value for non-terminal states
                non_terminal_mask = (dones_tensor == 0)
                avg_next_state_value = next_q_max[non_terminal_mask].mean().item() if non_terminal_mask.any() else 0.0

            # Convert target list back to tensor
            try:
                target_q = torch.tensor(target_q_val_list, dtype=torch.float32, device=DEVICE)
            except Exception as e:
                print(f"Error converting target list to tensor: {e}")
                return 0.0, 0.0

        # --- Calculate Loss ---
        # Ensure target_q has the same shape as current_q
        if target_q.shape != current_q.shape:
            print(f"Error: Shape mismatch between current_q ({current_q.shape}) and target_q ({target_q.shape})")
            return 0.0, 0.0

        # Using Smooth L1 Loss (Huber Loss) - less sensitive to outliers than MSE
        loss = F.smooth_l1_loss(current_q, target_q.detach()) # Use detach() on target

        # --- Perform Gradient Descent ---
        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(self.network.parameters(), max_norm=1.0)
        self.optimizer.step()

        # --- Update Target Network ---
        self.update_count += 1
        if self.update_count % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.network.state_dict())

        return loss.item(), avg_next_state_value # Return loss and average value of next states


# ============================================================================
# NETWORK GENERATION WITH ENHANCED METRICS AND REAL-WORLD DATASETS
# ============================================================================
def load_real_world_networks():
    """
    Loads real-world datasets from the 'datasets' directory.
    Returns a list of network dictionaries compatible with the synthetic networks format.
    """
    datasets_dir = 'datasets'
    real_world_networks = []
    
    # Define dataset files and their properties
    dataset_configs = [
        {
            'filename': 'polblogs.txt',
            'name': 'POLBLOGS',
            'description': 'Political blogs network',
            'delimiter': ' ',
            'has_edge_weights': False
        },
        {
            'filename': 'facebook.txt',
            'name': 'FACEBOOK',
            'description': 'Facebook social network',
            'delimiter': ' ',
            'has_edge_weights': False
        },
        {
            'filename': 'email-Eu-core.txt',
            'name': 'EMAIL',
            'description': 'Email communication network',
            'delimiter': ' ',
            'has_edge_weights': False
        }
    ]
    
    print(f"\n[LOADING REAL-WORLD DATASETS FROM '{datasets_dir}']")
    
    for config in dataset_configs:
        filepath = os.path.join(datasets_dir, config['filename'])
        
        if not os.path.exists(filepath):
            print(f"  ✗ Warning: Dataset file '{config['filename']}' not found. Skipping.")
            continue
            
        try:
            print(f"  -> Loading {config['name']} from {config['filename']}...")
            
            # Load the graph from edge list
            G = nx.read_edgelist(
                filepath,
                delimiter=config['delimiter'],
                nodetype=int,
                data=config['has_edge_weights']
            )
            
            # Ensure undirected
            if G.is_directed():
                G = G.to_undirected()
            
            # Remove self-loops
            G.remove_edges_from(nx.selfloop_edges(G))
            
            # Take largest connected component
            if not nx.is_connected(G):
                print(f"    -> Graph is disconnected. Taking largest connected component.")
                largest_cc = max(nx.connected_components(G), key=len)
                G = G.subgraph(largest_cc).copy()
            
            # Relabel nodes to consecutive integers starting from 0
            G = nx.convert_node_labels_to_integers(G, first_label=0)
            
            num_nodes = G.number_of_nodes()
            num_edges = G.number_of_edges()
            
            if num_nodes < 10:  # Skip very small networks
                print(f"    ✗ Skipping {config['name']}: Too small ({num_nodes} nodes).")
                continue
            
            # Calculate network metrics
            avg_clustering = nx.average_clustering(G)
            
            # Degree Entropy
            degree_sequence = [d for _, d in G.degree()]
            degree_counts = Counter(degree_sequence)
            probs = [count / num_nodes for count in degree_counts.values()]
            degree_entropy = -np.sum([p * np.log2(p) for p in probs if p > 0])
            
            # Structural Entropy (via Laplacian Eigenvalues)
            structural_entropy = 0.0
            try:
                L = nx.laplacian_matrix(G).astype(float).todense()
                eigenvalues = np.linalg.eigvalsh(L)
                eigenvalues = eigenvalues[eigenvalues > 1e-8]
                if eigenvalues.size > 0:
                    norm_eigenvalues = eigenvalues / np.sum(eigenvalues)
                    structural_entropy = -np.sum(norm_eigenvalues * np.log2(norm_eigenvalues + 1e-12))
            except Exception as e:
                print(f"    ! Could not compute structural entropy for {config['name']}: {e}")
            
            network_dict = {
                'name': config['name'],
                'graph': G,
                'phase': 'test',  # Real-world datasets are for testing only
                'num_nodes': num_nodes,
                'num_edges': num_edges,
                'avg_clustering': avg_clustering,
                'degree_entropy': degree_entropy,
                'structural_entropy': structural_entropy,
                'network_type': config['name'],  # Use dataset name as network type
                'source': 'real-world',
                'description': config['description']
            }
            
            real_world_networks.append(network_dict)
            print(f"    ✓ Loaded {config['name']}: {num_nodes} nodes, {num_edges} edges, Clustering: {avg_clustering:.4f}")
            
        except Exception as e:
            print(f"  ✗ Error loading {config['filename']}: {e}")
    
    print(f"  Total real-world networks loaded: {len(real_world_networks)}")
    return real_world_networks

def generate_optimized_networks():
    """
    Generates synthetic networks and calculates structural metrics.
    """
    print(f"\n[GENERATING OPTIMIZED NETWORKS]")
    networks = []
    # Configurations: (type, N, param, seed, phase)
    # BA: N, m; ER: N, p; WS: N, k, p
    configs = [
        ('BA', 800, 3, 42, 'train'), ('BA', 850, 4, 43, 'train'),
        ('ER', 800, 0.01, 44, 'train'), ('ER', 850, 0.009, 45, 'train'),
        ('WS', 800, 8, 0.1, 46, 'train'), ('WS', 850, 10, 0.1, 47, 'train'),
        ('BA', 1200, 3, 48, 'test'), ('ER', 1200, 0.008, 49, 'test'), # Adjusted p for ER-1200
        ('WS', 1200, 8, 0.1, 50, 'test'),
    ]

    for config in configs:
        G = None
        net_type = config[0]
        phase = config[-1]
        seed = config[-2] # Seed is second to last

        try:
            if net_type == 'BA':
                _, N, m, _, _ = config # Extract N, m
                G = nx.barabasi_albert_graph(N, m, seed=seed)
                name = f"BA-{N}-m{m}"
            elif net_type == 'ER':
                _, N, p, _, _ = config # Extract N, p
                G = nx.erdos_renyi_graph(N, p, seed=seed)
                name = f"ER-{N}-p{p:.3f}"
            elif net_type == 'WS':
                _, N, k, p, _, _ = config # Extract N, k, p (p is 4th element)
                G = nx.watts_strogatz_graph(N, k, p, seed=seed)
                name = f"WS-{N}-k{k}-p{p:.1f}" # Include p in name
            else:
                print(f"Warning: Unknown network type '{net_type}'. Skipping.")
                continue

            if G is None: continue # Skip if generation failed

            # --- Graph Cleaning and Preprocessing ---
            if G.is_directed(): G = G.to_undirected()
            G = nx.Graph(G) # Ensure it's a simple graph copy
            G.remove_edges_from(nx.selfloop_edges(G)) # Remove self-loops

            # Ensure graph is connected (use largest component)
            if not nx.is_connected(G):
                largest_cc = max(nx.connected_components(G), key=len, default=None)
                if largest_cc is None or len(largest_cc) < 2: # Skip if no suitable component
                    print(f"Warning: Graph {name} is disconnected or too small after cleaning. Skipping.")
                    continue
                G = G.subgraph(largest_cc).copy() # Work on a copy

            # Relabel nodes to be consecutive integers starting from 0
            G = nx.convert_node_labels_to_integers(G, first_label=0)
            G.graph['name'] = name # Store name within graph object

            num_nodes = G.number_of_nodes()
            if num_nodes == 0: continue # Skip empty graphs

            # --- Calculate Network Metrics ---
            avg_clustering = nx.average_clustering(G)

            # Degree Entropy
            degree_sequence = [d for _, d in G.degree()]
            if not degree_sequence: degree_entropy = 0.0
            else:
                degree_counts = Counter(degree_sequence)
                probs = [count / num_nodes for count in degree_counts.values()]
                degree_entropy = -np.sum([p * np.log2(p) for p in probs if p > 0])

            # Structural Entropy (via Laplacian Eigenvalues)
            structural_entropy = 0.0 # Default value
            try:
                L = nx.laplacian_matrix(G).astype(float).todense()
                eigenvalues = np.linalg.eigvalsh(L) # Eigenvalues of symmetric matrix
                eigenvalues = eigenvalues[eigenvalues > 1e-8] # Exclude zero eigenvalue(s)
                if eigenvalues.size > 0:
                    # Normalize eigenvalues (like probabilities)
                    norm_eigenvalues = eigenvalues / np.sum(eigenvalues)
                    structural_entropy = -np.sum(norm_eigenvalues * np.log2(norm_eigenvalues + 1e-12))
            except Exception as e:
                print(f"Warning: Could not compute structural entropy for {name}: {e}. Setting to 0.")

            networks.append({
                'name': name, 'graph': G, 'phase': phase,
                'num_nodes': num_nodes, 'num_edges': G.number_of_edges(),
                'avg_clustering': avg_clustering,
                'degree_entropy': degree_entropy,
                'structural_entropy': structural_entropy,
                'network_type': net_type,  # BA, ER, or WS
                'source': 'synthetic',
                'description': f'{net_type} synthetic network'
            })

        except Exception as e:
            print(f"Error generating or processing network with config {config}: {e}")

    # Load real-world datasets
    real_world_networks = load_real_world_networks()
    
    # Add real-world networks to the list
    networks.extend(real_world_networks)
    
    train_nets = [n for n in networks if n['phase'] == 'train']
    test_nets = [n for n in networks if n['phase'] == 'test']

    print(f"\nGenerated {len(networks)} total networks:")
    print(f"  - Synthetic Train: {len(train_nets)}")
    print(f"  - Synthetic Test: {len([n for n in test_nets if n['source'] == 'synthetic'])}")
    print(f"  - Real-World Test: {len([n for n in test_nets if n['source'] == 'real-world'])}")
    
    return train_nets, test_nets


# ============================================================================
# OPTIMIZED TRAINING AND EVALUATION
# ============================================================================
def optimized_train_agents(train_data, num_episodes=200, max_rounds=MAX_ROUNDS,
                           blocker_policy_type='minimax', adversary_policy_type='independent',
                           results_dir='results', lr=1e-4, gamma=0.95,
                           batch_size=64, buffer_size=20000, target_update_freq=100,
                           epsilon_start=0.5, epsilon_end=0.01, epsilon_decay=0.995):
    """ Trains Blocker and Adversary agents using specified policies. """

    print(f"\n[OPTIMIZED TRAINING - ADVERSARY FIRST - {num_episodes} EPISODES]")
    print(f"  Blocker: {blocker_policy_type.upper()} | Adversary: {adversary_policy_type.upper()}")
    print(f"  Learning Rate: {lr}, Gamma: {gamma}, Batch Size: {batch_size}, Buffer: {buffer_size}")
    print(f"  Target Update Freq: {target_update_freq}, Epsilon: {epsilon_start}->{epsilon_end} (decay={epsilon_decay})")

    # Initialize agents
    blocker = OptimizedMetaStrategyDQNAgent(
        is_blocker=True, use_minimax=(blocker_policy_type == 'minimax'),
        lr=lr, gamma=gamma, target_update_freq=target_update_freq,
        batch_size=batch_size, buffer_size=buffer_size
    )
    adversary = OptimizedMetaStrategyDQNAgent(
        is_blocker=False, use_minimax=(adversary_policy_type == 'minimax'),
        lr=lr, gamma=gamma, target_update_freq=target_update_freq,
        batch_size=batch_size, buffer_size=buffer_size
    )

    training_logs = [] # Stores episode summary data
    epsilon = epsilon_start

    pbar = tqdm(range(num_episodes), desc="Training", ncols=100)

    for episode in pbar:
        # Select a random graph for the episode
        graph_data = random.choice(train_data)
        env = OptimizedCompetitiveLTEnvironment(graph_data)
        try:
            state = env.reset()
        except Exception as e:
            print(f"Error resetting environment for {graph_data['name']} in episode {episode}: {e}. Skipping episode.")
            continue

        if state.num_nodes < 2: # Skip trivial graphs
            continue

        episode_transitions_b = [] # Temp storage for blocker transitions this episode
        episode_transitions_a = [] # Temp storage for adversary transitions

        previous_influence_abs = 0.0 # Store absolute influence for reward calc

        # --- Run single episode ---
        for round_num in range(max_rounds):
            env.current_round = round_num # Update env's round counter
            current_state_pyg = env.get_state() # Get state AFTER round update

            # --- Adversary's Turn ---
            try:
                adv_strategy_id, _, _, _, _ = adversary.select_action(current_state_pyg, blocker, epsilon)
                adv_node_idx = env.execute_meta_strategy(adv_strategy_id, player=0)
                env.select_seed(player=0, node=adv_node_idx) # Update env state
            except Exception as e:
                print(f"Error during Adversary turn (Ep {episode}, Rd {round_num}): {e}. Skipping round.")
                break # End episode if agent action fails

            # State changes after adversary moves, Blocker observes this new state
            blocker_observe_state_pyg = env.get_state()

            # --- Blocker's Turn ---
            try:
                blocker_strategy_id, _, _, _, _ = blocker.select_action(blocker_observe_state_pyg, adversary, epsilon)
                blocker_node_idx = env.execute_meta_strategy(blocker_strategy_id, player=1)
                env.select_seed(player=1, node=blocker_node_idx) # Update env state again
            except Exception as e:
                print(f"Error during Blocker turn (Ep {episode}, Rd {round_num}): {e}. Skipping round.")
                break # End episode if agent action fails

            # --- Propagation and Reward ---
            # Propagate influence after *both* players have moved in the round
            current_influence_abs = env.fast_propagate_lt(num_simulations=3) # Use fewer simulations for speed during training
            done = (round_num == max_rounds - 1)

            # Calculate rewards (continuous zero-sum)
            adv_reward, blocker_reward = env.calculate_rewards(previous_influence_abs, done)

            # Get the final state for this transition (after propagation, before next round's moves)
            next_state_pyg = env.get_state()

            # --- Store Transitions ---
            # Adversary transition: (state_before_adv_move, adv_action, adv_reward, state_after_blocker_move_and_prop)
            adversary.store_transition(current_state_pyg, adv_strategy_id, adv_reward, next_state_pyg, done)
            # Blocker transition: (state_before_blocker_move, blocker_action, blocker_reward, state_after_blocker_move_and_prop)
            blocker.store_transition(blocker_observe_state_pyg, blocker_strategy_id, blocker_reward, next_state_pyg, done)

            previous_influence_abs = current_influence_abs # Update for next round's reward calc

            if done: break # End episode loop

        # --- End of Episode ---

        # Update networks (perform gradient descent using sampled batches)
        # Pass opponent agent for Minimax target calculation
        b_loss, b_val = blocker.update(adversary)
        a_loss, a_val = adversary.update(blocker)

        # Epsilon decay
        epsilon = max(epsilon_end, epsilon * epsilon_decay)

        # --- Logging ---
        final_influence_abs = env.last_influenced_size
        final_norm_influence = final_influence_abs / env.num_nodes if env.num_nodes > 0 else 0.0
        # For analysis, determine 'win' based on < 50% threshold
        blocker_analysis_win = 1 if final_norm_influence < 0.5 else 0

        # Get final rewards (calculated in the last step's calculate_rewards call if done=True)
        _, final_blocker_reward = env.calculate_rewards(previous_influence_abs, done=True)

        training_logs.append({
            'episode': episode, 'network_name': graph_data['name'],
            'final_influence_abs': final_influence_abs,
            'norm_influence': final_norm_influence,
            'blocker_analysis_win': blocker_analysis_win,
            'final_blocker_reward': final_blocker_reward,
            'blocker_loss': b_loss, 'adversary_loss': a_loss,
            'avg_q_val_blocker': b_val, 'avg_q_val_adversary': a_val,
            'epsilon': epsilon
        })

        # Update progress bar description
        if episode % 20 == 0 and len(training_logs) >= 20:
            recent_logs = training_logs[-20:]
            avg_win_rate = np.mean([log['blocker_analysis_win'] for log in recent_logs]) * 100
            avg_norm_inf = np.mean([log['norm_influence'] for log in recent_logs]) * 100
            avg_b_loss = np.mean([log['blocker_loss'] for log in recent_logs])
            avg_a_loss = np.mean([log['adversary_loss'] for log in recent_logs])
            pbar.set_postfix({
                'B_Win%': f'{avg_win_rate:.1f}', # Analysis win rate (<50%)
                'E[I]%': f'{avg_norm_inf:.1f}',
                'B_Loss': f'{avg_b_loss:.3f}',
                'A_Loss': f'{avg_a_loss:.3f}',
                'Eps': f'{epsilon:.3f}'
            })

        # Optional: Clear GPU cache periodically if memory is an issue
        if episode % 50 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --- End of Training ---
    pbar.close()

    # Save training logs
    os.makedirs(results_dir, exist_ok=True)
    log_df = pd.DataFrame(training_logs)
    log_df.to_csv(f'{results_dir}/training_data.csv', index=False)
    print(f"\nSaved training logs to: {results_dir}/training_data.csv")

    return blocker, adversary, training_logs


def optimized_evaluate_agents(blocker, adversary, train_networks, test_networks,
                              max_rounds=MAX_ROUNDS, num_games=5, results_dir='results',
                              propagation_sims=10): # More simulations for evaluation accuracy
    """ Evaluates trained agents with exploration turned off (epsilon=0). """

    print(f"\n[OPTIMIZED EVALUATION - ADVERSARY FIRST ({num_games} games/net)]")
    print(f"  Propagation Simulations: {propagation_sims}")

    all_networks = train_networks + test_networks # Evaluate on both train and test sets
    evaluation_results = [] # Stores final outcome per game
    strategy_log = []       # Stores strategy choices per round
    influence_trace_log = [] # Stores influence progression per round

    # Ensure agents are in evaluation mode
    blocker.network.eval()
    adversary.network.eval()

    for graph_data in tqdm(all_networks, desc="Evaluating Networks", ncols=100):
        for game_num in range(num_games):
            env = OptimizedCompetitiveLTEnvironment(graph_data)
            try:
                state = env.reset()
            except Exception as e:
                print(f"Error resetting environment for {graph_data['name']} during evaluation game {game_num}: {e}. Skipping game.")
                continue

            if state.num_nodes < 2: continue

            previous_influence_abs = 0.0
            current_norm_influence = 0.0
            current_influence_abs = 0.0

            # --- Run single game (no exploration) ---
            for round_num in range(max_rounds):
                env.current_round = round_num
                current_state_pyg = env.get_state()

                # Adversary move (epsilon=0)
                try:
                    adv_strategy_id, _, adv_q_values, _, adv_value = adversary.select_action(current_state_pyg, blocker, epsilon=0)
                    adv_node_idx = env.execute_meta_strategy(adv_strategy_id, player=0)
                    env.select_seed(player=0, node=adv_node_idx)
                except Exception as e:
                    print(f"Eval Error (Adv): Ep {game_num}, Rd {round_num}, Net {graph_data['name']}: {e}")
                    break

                blocker_observe_state_pyg = env.get_state()

                # Blocker move (epsilon=0)
                try:
                    blocker_strategy_id, _, blocker_q_values, _, blocker_value = blocker.select_action(blocker_observe_state_pyg, adversary, epsilon=0)
                    blocker_node_idx = env.execute_meta_strategy(blocker_strategy_id, player=1)
                    env.select_seed(player=1, node=blocker_node_idx)
                except Exception as e:
                    print(f"Eval Error (Blocker): Ep {game_num}, Rd {round_num}, Net {graph_data['name']}: {e}")
                    break

                # Propagate influence (use evaluation simulation count)
                current_influence_abs = env.fast_propagate_lt(num_simulations=propagation_sims)
                current_norm_influence = current_influence_abs / env.num_nodes if env.num_nodes > 0 else 0.0
                current_norm_influence = np.clip(current_norm_influence, 0.0, 1.0) # Ensure valid range

                done = (round_num == max_rounds - 1)

                # --- Log round data ---
                strategy_log.append({
                    'network': graph_data['name'], 'phase': graph_data['phase'],
                    'game_num': game_num, 'round': round_num,
                    'blocker_strategy_id': blocker_strategy_id,
                    'adversary_strategy_id': adv_strategy_id,
                    'blocker_q_max': np.max(blocker_q_values) if blocker_q_values is not None else np.nan,
                    'adversary_q_max': np.max(adv_q_values) if adv_q_values is not None else np.nan,
                    'norm_influence': current_norm_influence,
                    'network_type': graph_data.get('network_type', 'Unknown'),
                    'source': graph_data.get('source', 'synthetic')
                })

                influence_trace_log.append({
                    'network': graph_data['name'], 'phase': graph_data['phase'],
                    'game_num': game_num, 'round': round_num,
                    'norm_influence': current_norm_influence,
                    'abs_influence': current_influence_abs,
                    'network_type': graph_data.get('network_type', 'Unknown'),
                    'source': graph_data.get('source', 'synthetic')
                })

                previous_influence_abs = current_influence_abs
                if done: break

            # --- End of Game ---
            # Use last calculated influence as final
            final_influence_abs = current_influence_abs
            final_norm_influence = current_norm_influence

            # Calculate final rewards based on final state
            final_adv_reward, final_blocker_reward = env.calculate_rewards(previous_influence_abs, done=True)

            # Determine win based on < 50% threshold for analysis
            blocker_analysis_win = 1 if final_norm_influence < 0.5 else 0

            # --- Log game results ---
            evaluation_results.append({
                'network': graph_data['name'], 'phase': graph_data['phase'],
                'game_num': game_num, 'num_nodes': env.num_nodes,
                'final_influence_abs': final_influence_abs,
                'norm_influence': final_norm_influence,
                'adversary_seeds': len(env.occupied[0]),
                'blocker_seeds': len(env.occupied[1]),
                'blocker_analysis_win': blocker_analysis_win,
                'final_blocker_reward': final_blocker_reward,
                'final_adversary_reward': final_adv_reward,
                # Include network metrics used in analysis plots
                'avg_clustering': graph_data.get('avg_clustering', np.nan),
                'degree_entropy': graph_data.get('degree_entropy', np.nan),
                'structural_entropy': graph_data.get('structural_entropy', np.nan),
                'network_type': graph_data.get('network_type', 'Unknown'),
                'source': graph_data.get('source', 'synthetic'),
                'description': graph_data.get('description', '')
            })

    # --- Save Evaluation Results ---
    os.makedirs(results_dir, exist_ok=True)
    pd.DataFrame(evaluation_results).to_csv(f'{results_dir}/evaluation_results.csv', index=False)
    pd.DataFrame(strategy_log).to_csv(f'{results_dir}/evaluation_strategy_log.csv', index=False)
    pd.DataFrame(influence_trace_log).to_csv(f'{results_dir}/evaluation_trace_log.csv', index=False)
    print(f"\nSaved evaluation results to: {results_dir}/")

    return evaluation_results, strategy_log, influence_trace_log


# ============================================================================
# EXPERIMENT RUNNER
# ============================================================================
def run_optimized_experiment(exp_name, b_policy, a_policy, train_nets, test_nets,
                             num_episodes=700, max_rounds=MAX_ROUNDS):
    """ Runs a full training and evaluation cycle for one experiment configuration. """

    print("\n" + "=" * 80)
    print(f"RUNNING EXPERIMENT: {exp_name}")
    print(f"  Blocker Policy: {b_policy.upper()}")
    print(f"  Adversary Policy: {a_policy.upper()}")
    print(f"  Training Episodes: {num_episodes}")
    print("=" * 80)

    RESULTS_DIR = f"results_{exp_name}" # Specific directory for this experiment
    os.makedirs(RESULTS_DIR, exist_ok=True)

    # --- Train Agents ---
    blocker, adversary, training_logs = optimized_train_agents(
        train_nets,
        num_episodes=num_episodes,
        max_rounds=max_rounds,
        blocker_policy_type=b_policy,
        adversary_policy_type=a_policy,
        results_dir=RESULTS_DIR
    )

    # --- Evaluate Agents ---
    eval_results, eval_strategy_log, eval_trace_log = optimized_evaluate_agents(
        blocker,
        adversary,
        train_nets, # Evaluate on training nets
        test_nets,  # Evaluate on test nets
        max_rounds=max_rounds,
        num_games=5, # Number of evaluation games per network
        results_dir=RESULTS_DIR
    )

    print(f"\n--- COMPLETED EXPERIMENT: {exp_name} ---")
    print(f"  Results saved in: {os.path.abspath(RESULTS_DIR)}")
    
    # Save experiment summary
    summary = {
        'experiment_name': exp_name,
        'blocker_policy': b_policy,
        'adversary_policy': a_policy,
        'num_episodes': num_episodes,
        'max_rounds': max_rounds,
        'num_train_networks': len(train_nets),
        'num_test_networks': len(test_nets),
        'num_real_world_networks': len([n for n in test_nets if n.get('source') == 'real-world']),
        'training_records': len(training_logs),
        'evaluation_records': len(eval_results),
        'strategy_log_records': len(eval_strategy_log),
        'trace_log_records': len(eval_trace_log)
    }
    
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(f'{RESULTS_DIR}/experiment_summary.csv', index=False)
    
    return summary


# ============================================================================
# MAIN EXECUTION SCRIPT
# ============================================================================
if __name__ == "__main__":

    # Check if datasets directory exists
    if not os.path.exists('datasets'):
        print("\n⚠️  WARNING: 'datasets' directory not found!")
        print("  Please run the dataset download script first:")
        print("  It will create a 'datasets' directory with:")
        print("    - polblogs.txt")
        print("    - facebook.txt") 
        print("    - email-Eu-core.txt")
        print("\n  The script will continue with synthetic networks only.")
        print("  Real-world datasets will be skipped if not found.\n")

    # --- Part 1: Run Experiments ---
    print("--- PART 1: RUNNING EXPERIMENTS ---")
    train_networks, test_networks = generate_optimized_networks()

    # Check if network generation was successful
    if not train_networks or not test_networks:
        print("ERROR: Network generation failed. Exiting.")
        exit() # Exit if no networks were created

    # Reduce episodes for faster testing if needed, increase for full training
    NUM_EPISODES = 700 # Keep at 700 for comparable results
    print(f"Number of Training Episodes per Experiment: {NUM_EPISODES}")

    experiments_to_run = [
        # Exp Name                 Blocker Policy  Adversary Policy
        ("EXP1_Indep_vs_Indep",    'independent',  'independent'),
        ("EXP2_Minimax_vs_Indep",  'minimax',      'independent'),
        ("EXP3_Minimax_vs_Minimax",'minimax',      'minimax'),
        ("EXP4_Indep_vs_Minimax",  'independent',  'minimax')
    ]

    all_summaries = []
    
    for exp_name, b_policy, a_policy in experiments_to_run:
        print(f"\n>>> Starting Experiment: {exp_name} <<<")
        # Ensure networks are passed correctly
        summary = run_optimized_experiment(
            exp_name=exp_name,
            b_policy=b_policy,
            a_policy=a_policy,
            train_nets=train_networks, # Pass the list of dicts
            test_nets=test_networks,   # Pass the list of dicts
            num_episodes=NUM_EPISODES,
            max_rounds=MAX_ROUNDS
        )
        all_summaries.append(summary)

    # Save combined summary of all experiments
    combined_summary_df = pd.DataFrame(all_summaries)
    combined_summary_df.to_csv('all_experiments_summary.csv', index=False)
    
    print("\n" + "=" * 80)
    print("--- ALL EXPERIMENTS COMPLETED ---")
    print("Reward Model: Continuous Zero-Sum (R_B = 1 - 2*I_N)")
    print("Datasets: Synthetic (BA, ER, WS) + Real-World (Polblogs, Facebook, Email)")
    print("=" * 80)
    
    # Print final summary
    print("\n--- FINAL SUMMARY ---")
    print(f"Total experiments completed: {len(experiments_to_run)}")
    print(f"Results saved in directories: results_EXP1_Indep_vs_Indep, results_EXP2_Minimax_vs_Indep, etc.")
    print("\nEach experiment directory contains:")
    print("  - training_data.csv: Training logs (episode-level metrics)")
    print("  - evaluation_results.csv: Final game outcomes")
    print("  - evaluation_strategy_log.csv: Strategy choices per round")
    print("  - evaluation_trace_log.csv: Influence progression per round")
    print("  - experiment_summary.csv: Experiment configuration and counts")
    print("\nAll experiment summaries combined in: all_experiments_summary.csv")
    
    print("\n--- SCRIPT FINISHED ---")

INFLUENCE MINIMIZATION: ADVERSARY-FIRST VERSION (V6.1 - Continuous Reward)
WITH REAL-WORLD DATASETS: Polblogs, Facebook, Email-Eu-Core
Device: cuda | CUDA: True
MODEL: COMPETITIVE LINEAR THRESHOLD (LT)
REWARD: Continuous Zero-Sum (R_B = 1 - 2*I_N)
WIN CONDITION (for analysis): Blocker if I_N < 0.5 | Adversary if I_N > 0.5
--- PART 1: RUNNING EXPERIMENTS ---

[GENERATING OPTIMIZED NETWORKS]

[LOADING REAL-WORLD DATASETS FROM 'datasets']
  -> Loading POLBLOGS from polblogs.txt...
    -> Graph is disconnected. Taking largest connected component.
    ✓ Loaded POLBLOGS: 1222 nodes, 16714 edges, Clustering: 0.3203
  -> Loading FACEBOOK from facebook.txt...
    ✓ Loaded FACEBOOK: 4039 nodes, 88234 edges, Clustering: 0.6055
  -> Loading EMAIL from email-Eu-core.txt...
    -> Graph is disconnected. Taking largest connected component.
    ✓ Loaded EMAIL: 986 nodes, 16064 edges, Clustering: 0.4071
  Total real-world networks loaded: 3

Generated 12 total networks:
  - Synthetic Train: 6
  - Synth

Training: 100%|█| 700/700 [30:31<00:00,  2.62s/it, B_Win%=100.0, E[I]%=23.4, B_Loss=0.002, A_Loss=0.



Saved training logs to: results_EXP1_Indep_vs_Indep/training_data.csv

[OPTIMIZED EVALUATION - ADVERSARY FIRST (5 games/net)]
  Propagation Simulations: 10


Evaluating Networks: 100%|██████████████████████████████████████████| 12/12 [15:44<00:00, 78.73s/it]



Saved evaluation results to: results_EXP1_Indep_vs_Indep/

--- COMPLETED EXPERIMENT: EXP1_Indep_vs_Indep ---
  Results saved in: /kaggle/working/results_EXP1_Indep_vs_Indep

>>> Starting Experiment: EXP2_Minimax_vs_Indep <<<

RUNNING EXPERIMENT: EXP2_Minimax_vs_Indep
  Blocker Policy: MINIMAX
  Adversary Policy: INDEPENDENT
  Training Episodes: 700

[OPTIMIZED TRAINING - ADVERSARY FIRST - 700 EPISODES]
  Blocker: MINIMAX | Adversary: INDEPENDENT
  Learning Rate: 0.0001, Gamma: 0.95, Batch Size: 64, Buffer: 20000
  Target Update Freq: 100, Epsilon: 0.5->0.01 (decay=0.995)


...


Saved evaluation results to: results_EXP3_Minimax_vs_Minimax/

--- COMPLETED EXPERIMENT: EXP3_Minimax_vs_Minimax ---
  Results saved in: /kaggle/working/results_EXP3_Minimax_vs_Minimax

>>> Starting Experiment: EXP4_Indep_vs_Minimax <<<

RUNNING EXPERIMENT: EXP4_Indep_vs_Minimax
  Blocker Policy: INDEPENDENT
  Adversary Policy: MINIMAX
  Training Episodes: 700

[OPTIMIZED TRAINING - ADVERSARY FIRST - 700 EPISODES]
  Blocker: INDEPENDENT | Adversary: MINIMAX
  Learning Rate: 0.0001, Gamma: 0.95, Batch Size: 64, Buffer: 20000
  Target Update Freq: 100, Epsilon: 0.5->0.01 (decay=0.995)


Training: 100%|█| 700/700 [36:08<00:00,  3.10s/it, B_Win%=100.0, E[I]%=27.9, B_Loss=0.002, A_Loss=0.



Saved training logs to: results_EXP4_Indep_vs_Minimax/training_data.csv

[OPTIMIZED EVALUATION - ADVERSARY FIRST (5 games/net)]
  Propagation Simulations: 10


Evaluating Networks: 100%|██████████████████████████████████████████| 12/12 [16:10<00:00, 80.84s/it]



Saved evaluation results to: results_EXP4_Indep_vs_Minimax/

--- COMPLETED EXPERIMENT: EXP4_Indep_vs_Minimax ---
  Results saved in: /kaggle/working/results_EXP4_Indep_vs_Minimax

--- ALL EXPERIMENTS COMPLETED ---
Reward Model: Continuous Zero-Sum (R_B = 1 - 2*I_N)
Datasets: Synthetic (BA, ER, WS) + Real-World (Polblogs, Facebook, Email)

--- FINAL SUMMARY ---
Total experiments completed: 4
Results saved in directories: results_EXP1_Indep_vs_Indep, results_EXP2_Minimax_vs_Indep, etc.

Each experiment directory contains:
  - training_data.csv: Training logs (episode-level metrics)
  - evaluation_results.csv: Final game outcomes
  - evaluation_strategy_log.csv: Strategy choices per round
  - evaluation_trace_log.csv: Influence progression per round
  - experiment_summary.csv: Experiment configuration and counts

All experiment summaries combined in: all_experiments_summary.csv

--- SCRIPT FINISHED ---


In [4]:
import shutil

# Path to the folder you want to download
folder_path = '/kaggle/working/'
zip_path = '/kaggle/working/results.zip'

# Create a ZIP file
shutil.make_archive('/kaggle/working/results', 'zip', folder_path)

# Download the ZIP file
from IPython.display import FileLink
FileLink(zip_path)

/kaggle/working/results.zip